# Ingestão de Dados de Logística — Azure Data Lake

Este notebook realiza a ingestão dos arquivos de logística no Azure Data Lake.

**Fluxo:** Landing → Validação → Bronze → Registro de processamento

In [ ]:
# No terminal instale: pip install azure-storage-blob
#                      pip install python-dotenv

Criar pasta:

.env

.gitignore

## Obter a credencial no Azure

No Portal do Azure:

1. Abra sua conta de armazenamento **`stbiecommerce2026`**.
2. No menu lateral, procure **Security + networking → Access keys**.
3. Nessa tela haverá `key1` e `key2`.
4. Em **key1**, localize **Connection string** e clique em **Show**.
5. Copie e cole em .env

In [1]:
#testar o .env

import os
from dotenv import load_dotenv

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

if connection_string:
    print("Credencial carregada com sucesso!")
else:
    print("Credencial não encontrada.")

Credencial carregada com sucesso!


In [2]:
# conectar ao Azure
from azure.storage.blob import BlobServiceClient

blob_service = BlobServiceClient.from_connection_string(
    connection_string
)

print("Conexão com o Azure criada!")

Conexão com o Azure criada!


In [3]:
# listar os contêineres

for container in blob_service.list_containers():
    print(container["name"])

bronze
landing


In [4]:
# listar os arquivos da entrada

container_landing = blob_service.get_container_client("landing")

arquivos = container_landing.list_blobs(
    name_starts_with="logistica/entrada/"
)

for arquivo in arquivos:
    print(arquivo.name)

logistica/entrada/entregas_2025_T1.csv


In [5]:
#pegar o camonho

arquivos = container_landing.list_blobs(
    name_starts_with="logistica/entrada/"
)

for arquivo in arquivos:
    if arquivo.name.endswith(".csv"):
        caminho_arquivo = arquivo.name
        break

print(caminho_arquivo)

logistica/entrada/entregas_2025_T1.csv


In [6]:
# ler o CSV da Landing

import pandas as pd
from io import BytesIO

blob = container_landing.get_blob_client(caminho_arquivo)

dados = blob.download_blob().readall()

df = pd.read_csv(BytesIO(dados))

df.head()

,id_entrega,id_pedido,codigo_rastreio,transportadora,data_postagem,previsao_entrega,data_entrega,status_entrega,custo_frete,tentativas_entrega
0,2,2,BR0000000002C,Correios,2025-01-16,2025-01-23,2025-01-27,Entregue,29.72,1
1,4,4,BR0000000004J,Jadlog,2025-01-22,2025-01-26,2025-01-28,Entregue,29.54,1
2,7,7,BR0000000007J,Jadlog,2025-01-21,2025-01-28,2025-01-28,Entregue,26.14,1
3,27,27,BR0000000027C,Correios,2025-02-01,2025-02-10,2025-02-08,Entregue,20.23,1
4,28,28,BR0000000028J,Jadlog,2025-02-20,2025-02-24,2025-02-27,Entregue,39.78,1


In [7]:
print("Quantidade de linhas:", df.shape[0])
print("Quantidade de colunas:", df.shape[1])

print("\nColunas:")
print(df.columns.tolist())

Quantidade de linhas: 4408
Quantidade de colunas: 10

Colunas:
['id_entrega', 'id_pedido', 'codigo_rastreio', 'transportadora', 'data_postagem', 'previsao_entrega', 'data_entrega', 'status_entrega', 'custo_frete', 'tentativas_entrega']


### Validações técnicas na Landing

| Validação | Exemplo | Se falhar |
|---|---|---|
| Arquivo existe | O CSV realmente chegou | Não processa |
| Extensão permitida | `.csv` | → `erros` |
| Arquivo não está vazio | Tamanho > 0 bytes | → `erros` |
| Arquivo pode ser aberto | CSV não está corrompido | → `erros` |
| Estrutura mínima | Possui cabeçalho/colunas | → `erros` |
| Arquivo já foi ingerido | Mesmo arquivo/hash não foi processado | Não duplica |
| Destino identificável | Ex.: arquivo de 2025 → `bronze/logistica/2025` | → `erros` |

In [8]:
import os
import pandas as pd
from io import BytesIO

nome_arquivo = os.path.basename(caminho_arquivo)

erros = []

# 1. Verificar extensão
if not nome_arquivo.lower().endswith(".csv"):
    erros.append("O arquivo não é CSV.")

# 2. Verificar se o arquivo está vazio
if len(dados) == 0:
    erros.append("O arquivo está vazio.")

# 3. Verificar se o CSV pode ser lido
try:
    df = pd.read_csv(BytesIO(dados))
except Exception as erro:
    erros.append(f"Não foi possível ler o CSV: {erro}")

# Resultado
if len(erros) == 0:
    print("Arquivo válido para ingestão na Bronze.")
else:
    print("Arquivo rejeitado:")

    for erro in erros:
        print("-", erro)

Arquivo válido para ingestão na Bronze.


In [9]:
# Landing → Bronze

container_bronze = blob_service.get_container_client("bronze")

destino = "logistica/2025/" + nome_arquivo

blob_bronze = container_bronze.get_blob_client(destino)

blob_bronze.upload_blob(
    dados,
    overwrite=False
)

print("Arquivo enviado para a Bronze com sucesso!")
print("Destino:", destino)

Arquivo enviado para a Bronze com sucesso!
Destino: logistica/2025/entregas_2025_T1.csv


In [10]:
# Criar o registro em processados

import json
from datetime import datetime

registro = {
    "arquivo": nome_arquivo,
    "status": "sucesso",
    "origem": caminho_arquivo,
    "destino": destino,
    "data_processamento": datetime.now().isoformat()
}

nome_log = nome_arquivo.replace(".csv", ".json")

blob_log = container_landing.get_blob_client(
    f"logistica/processados/{nome_log}"
)

blob_log.upload_blob(
    json.dumps(registro, indent=4),
    overwrite=False
)

print("Registro de processamento criado.")

Registro de processamento criado.


In [11]:
# excluir arquivo da entrada

blob_entrada = container_landing.get_blob_client(caminho_arquivo)

blob_entrada.delete_blob()

print("Arquivo removido da Landing/entrada.")

Arquivo removido da Landing/entrada.
